<a href="https://colab.research.google.com/github/vicemnovo/TCC_ModeloNeural/blob/Fase-1/Fase%201%20-%20VIT-Tiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1 — Detector de Imagens Sintéticas (ViT-Tiny + FFT)

```
FASE 1 — Detector de Imagens Sintéticas (ViT-Tiny + FFT, implementação do zero)
Projeto Integrador — Detecção de Mídias Sintéticas Geradas por IA

Como usar no Google Colab:
1. Abra um novo notebook, ative GPU (Ambiente de execução > Alterar tipo de ambiente > GPU).
2. Faça upload deste arquivo OU copie cada bloco "# %%" para uma célula separada.
3. Rode as células em ordem.

Datasets:
- 140k Real and Fake Faces (Kaggle: xhlulu/140k-real-and-fake-faces) -> classes real/fake
- COCO 2017 (val2017, subconjunto) -> usado como imagens REAIS adicionais no treino,
  para reduzir o viés de detecção exclusivamente facial (conforme metodologia do artigo)

IMPORTANTE / PREMISSAS assumidas (avise se quiser mudar):
- COCO entra SOMENTE no split de treino (não tem rótulo fake correspondente, então não
  faz sentido incluí-lo em validação/teste, cujo objetivo é medir acurácia real x fake).
- Uso val2017 do COCO (≈1GB) em vez do train2017 (≈18GB), por restrição de espaço/tempo no Colab.
  Se quiser usar mais imagens, é só trocar a URL de download.
- Classificação binária (real=0, fake=1) com CrossEntropyLoss de 2 classes.
- ViT-Tiny com patch_size=16, embed_dim=192, depth=12, heads=3 -> ~5,7M parâmetros,
  batendo com o valor citado na Introdução do artigo.
```

### Instalação de dependências

In [ ]:
!pip install -q kaggle scikit-learn

### Configuração da API do Kaggle (via variáveis de ambiente)

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("UsuarioKaggle")
os.environ["KAGGLE_KEY"] = userdata.get("ChaveKaggle")

### Validação da conexão com o Kaggle

In [ ]:
def validar_conexao_kaggle():
    """
    Autentica com a API do Kaggle e faz uma chamada leve (listar datasets)
    para confirmar que as credenciais estão corretas antes de baixar os datasets.
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError:
        raise RuntimeError(
            "Biblioteca 'kaggle' não encontrada. Rode a Célula 1 (pip install) antes desta."
        )

    if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
        raise RuntimeError(
            "KAGGLE_USERNAME/KAGGLE_KEY não definidos. Preencha a Célula 2 com suas credenciais."
        )
    if os.environ["KAGGLE_USERNAME"] == "seu_usuario_kaggle":
        raise RuntimeError(
            "Você esqueceu de trocar o usuário/chave de exemplo na Célula 2 pelos seus dados reais."
        )

    api = KaggleApi()
    try:
        api.authenticate()
        # chamada leve só para confirmar que a autenticação foi aceita pelo servidor
        _ = api.dataset_list(search="titanic")
    except Exception as e:
        raise RuntimeError(
            f"Falha ao autenticar/conectar com a API do Kaggle. "
            f"Verifique usuário/chave e sua conexão. Detalhe: {e}"
        )

    print(f"✅ Conexão com o Kaggle validada com sucesso (usuário: {os.environ['KAGGLE_USERNAME']}).")


validar_conexao_kaggle()

✅ Conexão com o Kaggle validada com sucesso (usuário: joicecelestino).


### Download dos datasets

In [ ]:
os.makedirs("/content/data", exist_ok=True)

# 140k Real and Fake Faces
os.system(
    "kaggle datasets download -d xhlulu/140k-real-and-fake-faces "
    "-p /content/data/faces --unzip"
)

# COCO 2017 (subconjunto = val2017, ~5000 imagens, todas rotuladas como REAIS)
os.system("wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/data/coco_val2017.zip")
os.system("unzip -q /content/data/coco_val2017.zip -d /content/data/coco")

print("Download concluído. Verifique as pastas em /content/data")

Download concluído. Verifique as pastas em /content/data


### Imports gerais

In [ ]:
import io
import random
import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

torch.set_float32_matmul_precision("high")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMG_SIZE = 224

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB


### Localizar as pastas do dataset de faces

In [ ]:
# O dataset xhlulu/140k-real-and-fake-faces normalmente extrai para:
#   /content/data/faces/real_vs_fake/real-vs-fake/{train,valid,test}/{real,fake}
# Se o Kaggle mudar a estrutura, ajuste FACES_ROOT abaixo.

FACES_ROOT = "/content/data/faces/real_vs_fake/real-vs-fake"
if not os.path.isdir(FACES_ROOT):
    # fallback: procura automaticamente a pasta que contém train/valid/test
    for root, dirs, _ in os.walk("/content/data/faces"):
        if {"train", "valid", "test"}.issubset(set(dirs)):
            FACES_ROOT = root
            break
print("Usando FACES_ROOT =", FACES_ROOT)

COCO_DIR = "/content/data/coco/val2017"

Usando FACES_ROOT = /content/data/faces/real_vs_fake/real-vs-fake


### Transform de compressão JPEG simulada (data augmentation)

In [ ]:
class SimulateJPEGCompression:
    """Aplica compressão JPEG com qualidade aleatória, simulando degradação pós-captura."""

    def __init__(self, quality_range=(30, 90), p=0.5):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    SimulateJPEGCompression(quality_range=(30, 90), p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

### Dataset do COCO (todas as imagens rotuladas como REAL = 0)

In [ ]:
class CocoRealDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_images=None):
        self.root_dir = root_dir
        self.transform = transform
        files_ = [f for f in os.listdir(root_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        files_.sort()
        if max_images is not None:
            files_ = files_[:max_images]
        self.files = files_

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.files[idx])
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = 0  # real
        return img, label

### Montagem dos datasets de treino/validação/teste

In [ ]:
# Faces: assume estrutura ImageFolder com subpastas 'real' e 'fake'
train_faces = ImageFolder(os.path.join(FACES_ROOT, "train"), transform=train_transform)
valid_faces = ImageFolder(os.path.join(FACES_ROOT, "valid"), transform=eval_transform)
test_faces = ImageFolder(os.path.join(FACES_ROOT, "test"), transform=eval_transform)

# Garante que o índice 'fake' = 1 e 'real' = 0 (ImageFolder ordena alfabeticamente:
# fake < real, então por padrão fake=0, real=1 -> por isso remapeamos abaixo)
print("class_to_idx (faces):", train_faces.class_to_idx)


class RemapLabels(Dataset):
    """Remapeia os rótulos do ImageFolder para o padrão do projeto: real=0, fake=1."""

    def __init__(self, base_dataset, class_to_idx):
        self.base_dataset = base_dataset
        # cria o mapeamento a partir do class_to_idx original do ImageFolder
        self.remap = {}
        for class_name, orig_idx in class_to_idx.items():
            target = 0 if "real" in class_name.lower() else 1
            self.remap[orig_idx] = target

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, orig_label = self.base_dataset[idx]
        return img, self.remap[orig_label]


train_faces = RemapLabels(train_faces, train_faces.dataset.class_to_idx if hasattr(train_faces, "dataset") else ImageFolder(os.path.join(FACES_ROOT, "train")).class_to_idx)
valid_faces = RemapLabels(valid_faces, ImageFolder(os.path.join(FACES_ROOT, "valid")).class_to_idx)
test_faces = RemapLabels(test_faces, ImageFolder(os.path.join(FACES_ROOT, "test")).class_to_idx)

# COCO entra só no treino, como reais adicionais (diversidade semântica)
coco_train = CocoRealDataset(COCO_DIR, transform=train_transform, max_images=5000)

train_dataset = ConcatDataset([train_faces, coco_train])
valid_dataset = valid_faces
test_dataset = test_faces

print(f"Treino: {len(train_dataset)} | Validação: {len(valid_dataset)} | Teste: {len(test_dataset)}")

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

class_to_idx (faces): {'fake': 0, 'real': 1}
Treino: 105000 | Validação: 20000 | Teste: 20000


### ViT-Tiny implementado do zero

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                # (B, embed_dim, H/patch, W/patch)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        return x


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # cada um: (B, heads, N, head_dim)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        return out


class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLPBlock(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTTiny(nn.Module):
    """
    ViT-Tiny implementado do zero.
    Config: patch_size=16, embed_dim=192, depth=12, num_heads=3 -> ~5.7M parâmetros.
    Retorna o embedding do token [CLS] (não a classificação final), para permitir
    a fusão posterior com o branch de FFT.
    """

    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                     # (B, N, D)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)        # (B, N+1, D)
        x = x + self.pos_embed
        x = self.pos_dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        return x[:, 0]  # embedding do token [CLS]

### Branch de análise em frequência (FFT)

In [ ]:
class FFTBranch(nn.Module):
    """
    Extrai o espectro de magnitude via FFT 2D e aprende features com uma CNN pequena.
    Entrada: imagem RGB (B, 3, H, W) -> log-magnitude do espectro -> CNN -> vetor de features.
    """

    def __init__(self, in_channels=3, out_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, out_dim)

    @staticmethod
    def to_log_magnitude_spectrum(x):
        # x: (B, C, H, W) em [-1, 1] (já normalizado)
        fft = torch.fft.fft2(x, norm="ortho")
        fft_shifted = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft_shifted)
        log_magnitude = torch.log1p(magnitude)
        return log_magnitude

    def forward(self, x):
        spec = self.to_log_magnitude_spectrum(x)
        feat = self.conv(spec)
        feat = self.pool(feat).flatten(1)
        feat = self.fc(feat)
        return feat

### Modelo híbrido Fase 1: ViT-Tiny + FFT

In [ ]:
class Phase1Detector(nn.Module):
    def __init__(self, vit_embed_dim=192, fft_out_dim=128, num_classes=2, dropout=0.2):
        super().__init__()
        self.vit = ViTTiny(embed_dim=vit_embed_dim)
        self.fft_branch = FFTBranch(out_dim=fft_out_dim)

        fusion_dim = vit_embed_dim + fft_out_dim
        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        vit_feat = self.vit(x)
        fft_feat = self.fft_branch(x)
        fused = torch.cat([vit_feat, fft_feat], dim=1)
        logits = self.classifier(fused)
        return logits


model = Phase1Detector().to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Total de parâmetros do modelo: {num_params:,} ({num_params / 1e6:.2f}M)")

Total de parâmetros do modelo: 5,598,530 (5.60M)


### Loop de treinamento

In [ ]:
# Ajustes em relação à primeira rodada (que ficou em 74,5% de acurácia, abaixo da meta
# de 85%): ViT treinado do zero precisa de mais épocas e de warmup de LR para convergir
# bem, já que não tem o viés indutivo espacial das CNNs. Mudanças aplicadas:
#   - warmup linear de LR + cosine decay (em vez de só cosine)
#   - mais épocas (40) com early stopping por F1 de validação (paciência = 6)
#   - label smoothing (0.1) na loss
#   - mixed precision (AMP) para reduzir o custo por época no Colab
#   - gradient clipping (evita picos de gradiente típicos de Transformers)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu"):
                logits = model(imgs)
                loss = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    avg_loss = total_loss / len(loader.dataset)
    return {"loss": avg_loss, "accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


def train_one_epoch(model, loader, optimizer, criterion, scaler, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu"):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)

    return total_loss / len(loader.dataset)


NUM_EPOCHS = 40
WARMUP_EPOCHS = 3
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOPPING_PATIENCE = 6

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def lr_lambda(epoch):
    # warmup linear nas primeiras WARMUP_EPOCHS épocas, depois cosine decay até o fim
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, (NUM_EPOCHS - WARMUP_EPOCHS))
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

best_val_f1 = -1.0
epochs_without_improvement = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler
    )

    val_metrics = evaluate(model, valid_loader)

    scheduler.step()

    elapsed = time.time() - t0
    print(
        f"[Epoch {epoch:02d}/{NUM_EPOCHS}] "
        f"lr={current_lr:.2e} | train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | val_f1={val_metrics['f1']:.4f} | "
        f"{elapsed:.1f}s"
    )
    history.append({"epoch": epoch, "train_loss": train_loss, "lr": current_lr, **val_metrics})

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "/content/phase1_best_model.pt")
        print("  -> novo melhor modelo salvo.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"  -> early stopping: sem melhora em {EARLY_STOPPING_PATIENCE} épocas consecutivas.")
            break

[Epoch 01/40] lr=1.00e-04 | train_loss=0.6435 | val_loss=0.5893 | val_acc=0.6834 | val_f1=0.6357 | 160.3s
  -> novo melhor modelo salvo.
[Epoch 02/40] lr=2.00e-04 | train_loss=0.5978 | val_loss=0.5675 | val_acc=0.7089 | val_f1=0.6659 | 156.3s
  -> novo melhor modelo salvo.
[Epoch 03/40] lr=3.00e-04 | train_loss=0.5786 | val_loss=0.5483 | val_acc=0.7225 | val_f1=0.7115 | 155.4s
  -> novo melhor modelo salvo.
[Epoch 04/40] lr=3.00e-04 | train_loss=0.5609 | val_loss=0.5228 | val_acc=0.7460 | val_f1=0.7551 | 155.1s
  -> novo melhor modelo salvo.
[Epoch 05/40] lr=2.99e-04 | train_loss=0.5438 | val_loss=0.4960 | val_acc=0.7577 | val_f1=0.7525 | 153.3s
[Epoch 06/40] lr=2.98e-04 | train_loss=0.5330 | val_loss=0.4994 | val_acc=0.7524 | val_f1=0.7741 | 154.8s
  -> novo melhor modelo salvo.
[Epoch 07/40] lr=2.95e-04 | train_loss=0.5228 | val_loss=0.4669 | val_acc=0.7781 | val_f1=0.7806 | 154.5s
  -> novo melhor modelo salvo.
[Epoch 08/40] lr=2.91e-04 | train_loss=0.5133 | val_loss=0.4627 | val_ac

### Avaliação final no conjunto de teste

In [ ]:
model.load_state_dict(torch.load("/content/phase1_best_model.pt"))
test_metrics = evaluate(model, test_loader)
print("\n=== Resultado final no conjunto de TESTE ===")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

META_ACURACIA = 0.85
if test_metrics["accuracy"] >= META_ACURACIA:
    print(f"\n✅ Meta mínima de {META_ACURACIA*100:.0f}% de acurácia atingida.")
else:
    print(f"\n⚠️ Meta mínima de {META_ACURACIA*100:.0f}% de acurácia NÃO atingida ainda.")


=== Resultado final no conjunto de TESTE ===
loss: 0.2927
accuracy: 0.8814
precision: 0.8396
recall: 0.9431
f1: 0.8883

✅ Meta mínima de 85% de acurácia atingida.


### Calibração via Temperature Scaling

In [ ]:
class TemperatureScaler(nn.Module):
    """
    Calibra as probabilidades de saída aprendendo um único escalar T (Guo et al., 2017).
    logits_calibrados = logits / T
    """

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        logits = self.model(x)
        return logits / self.temperature

    def fit(self, loader, max_iter=50, lr=0.01):
        self.model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in loader:
                imgs = imgs.to(device)
                logits = self.model(imgs)
                all_logits.append(logits.cpu())
                all_labels.append(labels)
        all_logits = torch.cat(all_logits).to(device)
        all_labels = torch.cat(all_labels).to(device)

        nll_criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=max_iter)

        def closure():
            optimizer.zero_grad()
            loss = nll_criterion(all_logits / self.temperature, all_labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        print(f"Temperatura calibrada: T = {self.temperature.item():.4f}")


temp_scaler = TemperatureScaler(model).to(device)
temp_scaler.fit(valid_loader)

torch.save({
    "model_state_dict": model.state_dict(),
    "temperature": temp_scaler.temperature.item(),
}, "/content/phase1_calibrated_model.pt")

print("\nModelo calibrado salvo em /content/phase1_calibrated_model.pt")

Temperatura calibrada: T = 1.0700

Modelo calibrado salvo em /content/phase1_calibrated_model.pt


### Função de inferência com probabilidade calibrada

In [ ]:
@torch.no_grad()
def predict_proba(model, temperature, image_tensor):
    """
    image_tensor: (1, 3, 224, 224) já normalizado com eval_transform.
    Retorna: probabilidade calibrada de a imagem ser FAKE (classe 1).
    """
    model.eval()
    image_tensor = image_tensor.to(device)
    logits = model(image_tensor) / temperature
    probs = F.softmax(logits, dim=1)
    return probs[0, 1].item()  # P(fake)


print("\nPipeline da Fase 1 pronto. Use predict_proba(model, temp_scaler.temperature, img_tensor) para inferência.")


Pipeline da Fase 1 pronto. Use predict_proba(model, temp_scaler.temperature, img_tensor) para inferência.


In [ ]:
#haiiiiiii, hoooowwwwww

Salvamento dos modelos e histórico no Google Drive

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

PASTA_MODELOS = "/content/drive/MyDrive/TCC_Final/modelos"
os.makedirs(PASTA_MODELOS, exist_ok=True)

shutil.copy(
    "/content/phase1_best_model.pt",
    f"{PASTA_MODELOS}/phase1_best_model.pt"
)

print("Modelo salvo no Google Drive com sucesso.")

Mounted at /content/drive
Modelo salvo no Google Drive com sucesso.
